# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [1]:
# Putting all the imports in their own little block at the top
import re
import Bio.SeqIO
import Bio.Seq

In [2]:
def identify_gene(seqread):
    """Identify whether a sequencing read matches HA or NA.

    Args:
        seqread: A nucleotide sequence in sequencing-read orientation.

    Returns:
        "HA" or "NA" if the corresponding gene ending is found,
        or None if neither ending is found.
    """
    seqread_upper = str(seqread).upper()
    seqread_original = str(
        Bio.Seq.Seq(seqread_upper).reverse_complement()
    )

    ha_end = "CCGGATTTGCATATAATGATGCACCAT"
    na_end = "CACGATAGATAAATAATAGTGCACCAT"

    if ha_end in seqread_original:
        return "HA"
    elif na_end in seqread_original:
        return "NA"
    else:
        return None


# Create a known HA sequence and put it in sequencing-read orientation
ha_sequence = (
    "CCGGATTTGCATATAATGATGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

ha_test_read = Bio.Seq.Seq(ha_sequence).reverse_complement()


# Create a known NA sequence and put it in sequencing-read orientation
na_sequence = (
    "CACGATAGATAAATAATAGTGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

na_test_read = Bio.Seq.Seq(na_sequence).reverse_complement()


# Positive tests: known reads should match the correct gene
assert identify_gene(ha_test_read) == "HA"
print("PASS: HA read identified as HA.")

assert identify_gene(na_test_read) == "NA"
print("PASS: NA read identified as NA.")

# Negative tests: reads should not match the wrong gene
assert identify_gene(ha_test_read) != "NA"
print("PASS: HA read not identified as NA.")

assert identify_gene(na_test_read) != "HA"
print("PASS: NA read not identified as HA.")

# Negative test: unrelated garbage should match neither gene
assert identify_gene("ACGTACGTACGT") is None
print("PASS: unrelated sequence returned None.")

# Read the FASTQ file into a list of sequencing records
seqreads = list(
    Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq")
)

# Start the read counters at zero
ha_reads = 0
na_reads = 0
unmatched_reads = 0

# Identify the gene for each read and update the appropriate counter
for read in seqreads:
    gene = identify_gene(read.seq)

    if gene == "HA":
        ha_reads += 1
    elif gene == "NA":
        na_reads += 1
    else:
        unmatched_reads += 1

print("Total reads:", len(seqreads))
print("Reads matching HA:", ha_reads)
print("Reads matching NA:", na_reads)
print("Reads matching neither gene:", unmatched_reads)

PASS: HA read identified as HA.
PASS: NA read identified as NA.
PASS: HA read not identified as NA.
PASS: NA read not identified as HA.
PASS: unrelated sequence returned None.
Total reads: 10000
Reads matching HA: 5409
Reads matching NA: 4122
Reads matching neither gene: 469


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [3]:
def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: A nucleotide sequence in sequencing-read orientation.
        bclen: Length of the barcode.
        upstream: Sequence immediately before the barcode
            in the forward orientation.

    Returns:
        The barcode in the forward orientation, as an uppercase string,
        or None if the expected pattern is not found.

    Example:
        >>> read_barcode("TTTTTTTTTTTTTTTTGCGGCCGCCT")
        'AAAAAAAAAAAAAAAA'
    """
    seqread_upper = str(seqread).upper()
    seqread_original = str(
        Bio.Seq.Seq(seqread_upper).reverse_complement()
    )

    barcode_pattern = re.compile(
        f"{upstream}(?P<barcode>[ACTGN]{{{bclen}}})$"
    )

    match = barcode_pattern.search(seqread_original)

    if match:
        return match.group("barcode")
    else:
        return None


# Positive test: a valid read should return its forward barcode
assert read_barcode("TTTTTTTTTTTTTTTTGCGGCCGCCT") == "AAAAAAAAAAAAAAAA"
print("PASS: valid barcode returned in forward orientation.")

# Negative test: a barcode that is too short should be rejected
assert read_barcode("TTTTGCGGCCGCCT") is None
print("PASS: short barcode rejected.")

# Negative test: an incorrect constant sequence should be rejected
assert read_barcode("TTTTTTTTTTTTTTTTGCGGGCCT") is None
print("PASS: incorrect constant sequence rejected.")

# Separate dictionaries store the valid barcode counts for each gene
ha_barcode_counts = {}
na_barcode_counts = {}

# Separate counters track reads without valid barcodes
ha_invalid_barcodes = 0
na_invalid_barcodes = 0

for read in seqreads:
    gene = identify_gene(read.seq)

    if gene == "HA":
        barcode = read_barcode(read.seq)

        if barcode is None:
            ha_invalid_barcodes += 1
        else:
            if barcode in ha_barcode_counts:
                ha_barcode_counts[barcode] += 1
            else:
                ha_barcode_counts[barcode] = 1

    elif gene == "NA":
        barcode = read_barcode(read.seq)

        if barcode is None:
            na_invalid_barcodes += 1
        else:
            if barcode in na_barcode_counts:
                na_barcode_counts[barcode] += 1
            else:
                na_barcode_counts[barcode] = 1


print("HA reads without a valid barcode:", ha_invalid_barcodes)
print("NA reads without a valid barcode:", na_invalid_barcodes)

print("Unique valid HA barcodes:", len(ha_barcode_counts))
print("Unique valid NA barcodes:", len(na_barcode_counts))

PASS: valid barcode returned in forward orientation.
PASS: short barcode rejected.
PASS: incorrect constant sequence rejected.
HA reads without a valid barcode: 163
NA reads without a valid barcode: 212
Unique valid HA barcodes: 281
Unique valid NA barcodes: 138


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [4]:
# Find the highest HA barcode count
ha_highest_count = 0

for barcode in ha_barcode_counts:
    count = ha_barcode_counts[barcode]

    if count > ha_highest_count:
        ha_highest_count = count


# Find the highest barcode counts by bruteforcing them bottom-up lol
if ha_highest_count == 0:
    print("No valid HA barcodes were found.")
else:
    for barcode in ha_barcode_counts:
        if ha_barcode_counts[barcode] == ha_highest_count:
            print("Most frequent HA barcode:", barcode)
            print("Number of reads:", ha_highest_count)

na_highest_count = 0

for barcode in na_barcode_counts:
    count = na_barcode_counts[barcode]

    if count > na_highest_count:
        na_highest_count = count


# Print the NA barcode or barcodes with that count
if na_highest_count == 0:
    print("No valid NA barcodes were found.")
else:
    for barcode in na_barcode_counts:
        if na_barcode_counts[barcode] == na_highest_count:
            print("Most frequent NA barcode:", barcode)
            print("Number of reads:", na_highest_count)

Most frequent HA barcode: CCCGACCCGACATTAA
Number of reads: 155
Most frequent NA barcode: ACCAGTTCTCCCCGGG
Number of reads: 152
